# CivicWater Sentinel — P2 · Step 3: GRU forecaster with TensorFlow on IBM Z

The main model of the ladder (design doc §3, Table 2): a **small GRU reads the last 12 hours of
every sensor and predicts the next reading of all 33 pressure sensors**. It follows the
organisers' *Fraud detection with LSTM* demo: sequences of `timesteps`, a Keras recurrent model,
checkpoints, evaluation, **ONNX export**, and the **`ibm-zdnn-plugin`** check and
benchmark on the Telum NNPA.

The layer settings respect the accelerator constraints in design doc §4 (sigmoid/tanh, no
clipping, `reset_after=True` = ONNX `linear_before_reset=1`). The architecture is defined once
in `sentinel/forecasting/gru.py` and shared by training and inference.

1. **Part A – train (off-platform, e.g. Colab with a GPU)**
2. **Part B – on LinuxONE**: load weights, x86↔s390x parity, zDNN plugin check, latency,
   ONNX check for the zDLC path, and the comparison of the whole ladder.

**Running in Colab (training off-platform, design doc §4).** Clone the repository, upload
`scada_2018.csv.gz`, `sensors.csv` and `leaks_2018.csv` from `shared/civicwater/data/v1/`, then
run this cell before the setup cell (the registry defaults to the repo's official
`config/sensors.battledim.json`, as on the VM):

```python
!git clone https://github.com/pedruck/Team-CodigoMaster_IBM-Z-Datathon-2026---Civic-Water-Sentinal.git /content/repo
!pip -q install xgboost onnx onnxscript torch
import os
os.environ["REPO"] = "/content/repo"
os.environ["SENTINEL_DATA_DIR"] = "/content/data"      # put the v1/ files in /content/data/v1/
os.environ["SENTINEL_MODELS_DIR"] = "/content/models"  # copy this folder to shared/civicwater/models/ on the VM
```

In [ ]:
# --- Setup (same for every P2 notebook) --------------------------------------------------
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")

# Repository root = first parent folder that contains src/sentinel (works on the VM and in Colab)
REPO = Path(os.environ.get("REPO") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sentinel").exists()))
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sentinel.forecasting import training as p2lib   # training helpers (tested in CI)
FAST = os.environ.get("P2_FAST") == "1"   # smoke-test mode: smaller training, same code path
MASK_POLICY = "reparados"   # P1's training mask (docs/data.md §7): repaired 2018 leaks excluded
print("repo   :", REPO)
print("data   :", p2lib.data_dir())
print("models :", p2lib.models_dir())
print("sensor registry:", os.environ.get("SENTINEL_SENSOR_REGISTRY") or p2lib.DEFAULT_REGISTRY, "| mask policy:", MASK_POLICY)
print("on LinuxONE (s390x):", p2lib.on_linuxone(), "| FAST mode:", FAST)

# Part A — Training

In [ ]:
import tensorflow as tf
from sentinel.forecasting.gru import MODEL_NAME, ONNX_FILE, WEIGHTS_FILE, build_gru_model
print("TensorFlow", tf.__version__)

WINDOW = 144            # 12 h of 5-minute steps (design doc: 12–24 h)
HORIZON = int(os.environ.get("P2_GRU_HORIZON", 1))  # 1 = next reading (design doc); see the note below
STRIDE = 12 if FAST else 3
UNITS = 16 if FAST else 64
EPOCHS = 1 if FAST else 10
BATCH = 256

layout = p2lib.load_layout()
df = p2lib.load_scada(2018, layout)
P = list(layout.pressure)
free = p2lib.clean_mask(df.index, MASK_POLICY)
n = p2lib.assert_feature_parity(df, layout, n=30 if FAST else 100)
print(f"feature parity OK on {n} rows")

## Sequences: `WINDOW` steps → pressures `HORIZON` steps later

`HORIZON = 1` is the design doc's "next reading". A one-step-ahead model sees the leak in its
own input a few minutes after onset and starts predicting the lower pressure, which shrinks the
residual CUSUM needs. The **leak contrast** below measures this; a larger `HORIZON` (the input
ends `HORIZON` steps before the target) keeps the leak out of the input for longer. The window
plus horizon must fit in the backend's history (`SENTINEL_HISTORY_CAPACITY`, 288 windows).

Inputs are standardised with Jan–Oct statistics. A target gets weight 1 only if its whole
input window *and* the target are clean under P1's mask, so the GRU learns from contiguous
clean stretches only (`docs/data.md` §7: the mask has gaps).

In [ ]:
feat = p2lib.gru_frame(df, layout).ffill().bfill()
target = df[P].ffill().bfill()
LAG = WINDOW + HORIZON - 1   # history windows the API must provide
weight = p2lib.sequence_weights(free & df[P].notna().all(axis=1), LAG)

tr_end = df.index.get_loc(pd.Timestamp(p2lib.TRAIN_END)) + 1
x_mean = feat.iloc[:tr_end].mean().to_numpy(np.float32)
x_std = feat.iloc[:tr_end].std().replace(0, 1).to_numpy(np.float32)
y_mean = target.iloc[:tr_end].mean().to_numpy(np.float32)
y_std = target.iloc[:tr_end].std().replace(0, 1).to_numpy(np.float32)
X = ((feat.to_numpy(np.float32) - x_mean) / x_std).astype(np.float32)
Y = ((target.to_numpy(np.float32) - y_mean) / y_std).astype(np.float32)
YW = np.c_[Y, weight].astype(np.float32)
print(f"usable training targets (clean stretches of {LAG + 1} windows): {weight[:tr_end].mean():.1%}")

def sequences(start, end, stride, shuffle):
    """Inputs X[t-LAG : t-HORIZON+1] → target at t, for targets t in [start, end)."""
    ds = tf.keras.utils.timeseries_dataset_from_array(
        X[start - LAG:end - HORIZON], YW[start:end], sequence_length=WINDOW,
        sequence_stride=stride, shuffle=shuffle, batch_size=BATCH, seed=1)
    return ds.map(lambda x, yw: (x, yw[:, :-1], yw[:, -1]))

assert LAG <= 288, "window + horizon must fit in SENTINEL_HISTORY_CAPACITY"
train_ds = sequences(LAG, tr_end, STRIDE, True)
val_ds = sequences(tr_end, len(df), STRIDE, False)
print("train batches:", int(train_ds.cardinality()), "| val batches:", int(val_ds.cardinality()))

## Model (shared architecture) and training with checkpoints

In [ ]:
model = build_gru_model(WINDOW, X.shape[1], len(P), UNITS)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
model.summary()

In [ ]:
ckpt_dir = Path("checkpoints/gru"); ckpt_dir.mkdir(parents=True, exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(ckpt_dir / "best.weights.h5"), save_weights_only=True, save_best_only=True),
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
pd.DataFrame(history.history).plot(figsize=(8, 3), title="loss (standardised MSE)"); plt.show()

## Validation in metres, compared with the ladder

In [ ]:
val_full = sequences(tr_end, len(df), 1, False)
pred = model.predict(val_full.map(lambda x, y, w: x), verbose=0) * y_std + y_mean
val_index = df.index[tr_end:tr_end + len(pred)]
resid = df.loc[val_index, P] - pd.DataFrame(pred, index=val_index, columns=P)
mae, sigma = p2lib.mae_by_sensor(resid, free), p2lib.robust_sigma(resid)
contrast = p2lib.leak_contrast(resid, sigma, free)
print(f"GRU validation MAE (clean windows): {mae.mean():.3f} m | leak contrast (pressure drops): {contrast}")
print("low-variance sensors (σ < 0.02 m):", p2lib.low_variance_sensors(sigma))

## Export: Keras weights, ONNX and a parity sample

* `model.weights.h5` — loaded by the API on LinuxONE with TensorFlow + `ibm-zdnn-plugin`.
* `model.onnx` — portable copy for the IBM Z Deep Learning Compiler (zDLC) path. The
  organisers' demos use tf2onnx, but with Keras 3 it turns the GRU into a generic `Loop`.
  `export_onnx` rebuilds the same network as `torch.nn.GRU` (identical maths) and exports **one
  ONNX `GRU` node with `linear_before_reset=1`**, the form zDLC can place on the NNPA.
* `parity_rows.csv` + `parity_expected_x86.csv` — checked again on s390x in Part B.

In [ ]:
from sentinel.forecasting.gru import export_onnx
VERSION = "0.1.0"
out = p2lib.models_dir() / f"{MODEL_NAME}@{VERSION}"
out.mkdir(parents=True, exist_ok=True)
model.save_weights(str(out / WEIGHTS_FILE))
onnx_info = export_onnx(model, out / ONNX_FILE)
print("ONNX:", onnx_info)
assert onnx_info["gru_nodes"] == 1 and onnx_info["linear_before_reset"] == 1

# parity sample: LAG+16 contiguous validation rows and the x86 forecasts for the last 16
k = tr_end + 200
rows = df.iloc[k - LAG - 16:k][list(layout.pressure) + list(layout.flow) + list(layout.tank_level)].ffill().bfill()
from sentinel.forecasting.gru import GRUForecaster
meta = p2lib.artefact_meta(
    MODEL_NAME, VERSION, layout, window=WINDOW, horizon=HORIZON, units=UNITS,
    feature_names=list(feat.columns),
    x_mean=x_mean.tolist(), x_std=x_std.tolist(), y_mean=y_mean.tolist(), y_std=y_std.tolist(),
    trained_on="2018 Jan-Oct, clean contiguous stretches (P1 mask); early-stopped on Nov-Dec 2018",
    training={"epochs_run": len(history.history["loss"]), "stride": STRIDE, "batch": BATCH},
    onnx=onnx_info,
    validation={"period": "2018-11-01..2018-12-31", "mae_m_mean": round(float(mae.mean()), 4),
                "mae_m": mae.round(4).to_dict(), "leak_contrast": contrast, "metrics_version": 2},
    sigma_m=sigma.round(5).to_dict(), mask_policy=MASK_POLICY,
)
p2lib.write_json(out / "meta.json", meta)
runtime = GRUForecaster(out)
ref = {}
for ts in rows.index[LAG:]:
    ref[ts] = runtime.predict(p2lib.build_request(rows, ts, layout, history=LAG)).expected_pressures
rows.to_csv(out / "parity_rows.csv")
pd.DataFrame(ref).T.rename_axis("timestamp").to_csv(out / "parity_expected_x86.csv")
print("exported to", out, "→", sorted(f.name for f in out.iterdir()))

# Part B — On IBM LinuxONE

Copy `models/gru-33@0.1.0/` to `shared/civicwater/models/` on the VM and run from here.

## IBM zDNN / NNPA check (same as the organisers' notebooks)

In [ ]:
import tensorflow as tf
report = p2lib.accelerator_report()
for k, v in report.items():
    print(f"{k:16}: {v}")
print("TensorFlow devices:", tf.config.list_physical_devices())
if not report["tf_zdnn_kernels"]:
    print("\n_z_ops.so not found: TensorFlow runs on the CPU here ('accelerator-ready').")

## Load the model exactly as the API does, and check x86 ↔ s390x parity

In [ ]:
from sentinel.forecasting.gru import GRUForecaster
from sentinel.forecasting.registry import load_forecaster
layout = p2lib.load_layout()
P = list(layout.pressure)
model_dir = p2lib.models_dir() / "gru-33@0.1.0"
gru = load_forecaster("gru", model_dir)
print(gru.info, "| history needed:", gru.min_history_windows, "windows")

rows = pd.read_csv(model_dir / "parity_rows.csv", index_col="timestamp", parse_dates=True)
ref = pd.read_csv(model_dir / "parity_expected_x86.csv", index_col="timestamp", parse_dates=True)
W = gru.min_history_windows
worst = 0.0
for ts in rows.index[W:]:
    out_ = gru.predict(p2lib.build_request(rows, ts, layout, history=W)).expected_pressures
    worst = max(worst, max(abs(out_[s] - ref.loc[ts, s]) for s in P))
print(f"max |{report['machine']} − x86| = {worst:.2e} m over {len(rows) - W} windows × {len(P)} sensors")

## Latency: one API forecast, and batch throughput (organisers' benchmark style)

The single-window number is what the alert path pays every 5 minutes (graph mode via
`tf.function`; an eager Keras call costs ~300 ms here). The batch number is the
throughput view used in the organisers' LSTM notebook.

In [ ]:
req = p2lib.build_request(rows, rows.index[W], layout, history=W)
single = p2lib.benchmark(lambda: gru.predict(req), repeats=30 if FAST else 100)
print("one window, 33 sensors:", {k: round(v, 3) for k, v in single.items()})

keras_model = gru.keras_model
BATCH_B = 256
n_features = len(p2lib.read_json(model_dir / "meta.json")["feature_names"])
xb = tf.constant(np.random.default_rng(0).normal(size=(BATCH_B, W, n_features)).astype("float32"))
batch = p2lib.benchmark(lambda: keras_model(xb, training=False).numpy(), repeats=10)
print(f"batch {BATCH_B}: median {batch['p50_ms']:.1f} ms → {BATCH_B / (batch['p50_ms'] / 1000):,.0f} sequences/s")

## ONNX artefact for the zDLC path

Checks the exported graph and the GRU attributes the IBM Z Deep Learning Compiler needs to place
the GRU on the NNPA (design doc §4).

In [ ]:
import onnx
om = onnx.load(str(model_dir / "model.onnx"))
onnx.checker.check_model(om)
nodes = [n for n in om.graph.node if n.op_type == "GRU"]
for n in nodes:
    a = {x.name: onnx.helper.get_attribute_value(x) for x in n.attribute}
    print("GRU node:", {k: (v.decode() if isinstance(v, bytes) else v) for k, v in a.items() if k != "activations"},
          "| activations:", [v.decode() for v in a.get("activations", [])])
assert len(nodes) == 1, "expected one fused GRU node for zDLC"

# Same numbers as the Keras model the API runs (ONNX reference evaluator, no onnxruntime needed)
from onnx.reference import ReferenceEvaluator
x1 = xb.numpy()[:2]
(onnx_out,) = ReferenceEvaluator(om).run(None, {"window": x1})
print("max |ONNX − Keras|:", float(np.abs(onnx_out - keras_model(x1, training=False).numpy()).max()))

## The model ladder on this machine

In [ ]:
meta = p2lib.read_json(model_dir / "meta.json")
meta.setdefault("latency_ms", {})[report["machine"]] = {"one_window": single, f"batch_{BATCH_B}": batch}
meta.setdefault("parity", {})[report["machine"]] = worst
p2lib.write_json(model_dir / "meta.json", meta)

def p50_of(lat):
    for key in ("one_window", "snapml auto + TreeSHAP (API default)"):
        if key in lat:
            return lat[key]["p50_ms"]
    return lat.get("p50_ms")

rows_ = []
for name, path in [("1 · baseline", "baseline-hourofweek@0.1.0.json"), ("2 · xgboost (Snap ML)", "xgboost-33@0.1.0/meta.json"), ("3 · GRU", "gru-33@0.1.0/meta.json")]:
    f = p2lib.models_dir() / path
    if not f.exists():
        continue
    m = p2lib.read_json(f)
    lat = m.get("latency_ms", {}).get(report["machine"], {})
    p50 = p50_of(lat) if lat else None
    rows_.append({"model": name, "val MAE (m)": m["validation"]["mae_m_mean"],
                  "leak contrast": m["validation"].get("leak_contrast"), "p50 latency (ms)": p50,
                  "metrics": "v2" if m["validation"].get("metrics_version") == 2 else "old: re-run"})
pd.DataFrame(rows_).set_index("model").round(3)

## Summary

* Each step must beat the previous one on validation MAE; latency per 5-minute window is far
  below the 300 s budget for every step.
* The API loads the GRU with `load_forecaster("gru", <dir>)`; it needs `WINDOW` windows of
  history (12 h), which matches `SENTINEL_HISTORY_CAPACITY`/`MIN_HISTORY_WINDOWS`.
* If the zDNN kernels are not present the model runs on the CPU and the result is reported as
  *accelerator-ready*, never as *accelerated* (design doc §5).